# CT.M1 - Data Acquisition
**Version 1.0.0 - February, 2026. Monterrey**

**Author:** Flavio F. Contreras-Torres


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/CheminformaticsTutorial/blob/main/01_Data_Acquisition/notebooks/01_pubchem_data_retrieval.ipynb)


---


### Contents

This notebook introduces the first stage of the computational workflow: retrieving molecular data from PubChem using its REST API.

The activities are structured to guide you through the complete process of:

1. Understanding the PubChem data model  
2. Querying compounds programmatically  
3. Retrieving molecular properties  
4. Constructing a structured dataset  
5. Performing initial data inspection  
6. Exporting a raw dataset for downstream curation  

This notebook is designed to be completed in approximately **45–60 minutes**.  

You are encouraged to modify queries, explore additional properties, and inspect returned data structures. The goal is not only to retrieve molecules, but to understand how chemical information is organized and accessed computationally.

---

### PubChem Data Retrieval  

**PubChem** is a public chemical information resource maintained by the NIH, providing access to molecular structures, identifiers, and associated chemical properties.

In this notebook, we will **search PubChem**, **retrieve molecular properties programmatically**, and **construct a structured dataset** suitable for downstream curation and analysis.

---

### Step 0: The Computational Environment
Before retrieving chemical data, we must load our specialized toolkit. In this module, we rely on a standard Python stack for web communication and data manipulation:

* **`requests`**: The industry standard for making HTTP calls. It allows our script to "talk" to the PubChem servers.
* **`pandas`**: A powerful library for data structures (DataFrames). It will serve as our digital lab notebook to organize molecular properties.
* **`os` & `time`**: Essential for file system management and ensuring we respect API rate limits (avoiding server overload).

**Note:** While **RDKit** is the heart of cheminformatics, we deliberately keep this first step "lightweight" to focus on raw data acquisition before moving into structural processing.

In [ ]:
import os
import time
import requests
import pandas as pd

### Step 1: Defining the Chemical Search Space
In this section, we define the search parameters. In a real-world project, this represents the **initial library selection**. We utilize a keyword-based search to identify a specific chemical class (e.g., flavonoids) and limit the output to ensure computational efficiency during this tutorial.

In [ ]:
# Step 1
BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
OUT_DIR = os.path.join("..", "data_raw")
os.makedirs(OUT_DIR, exist_ok=True)

QUERY = "flavonoid"   # cambia si quieres
MAX_CIDS = 200        # tamaño controlado para el tutorial

### Step 2: From Natural Language to Unique CIDs
Chemical names are often ambiguous or redundant. To ensure data integrity, we must convert our query into **PubChem CIDs**. The CID is a permanent, unique numerical index that acts as the "Social Security Number" for a molecule within the NIH ecosystem.

In [ ]:
# Step 2
def get_cids_by_name(query: str, max_cids: int = 200):
    # "name" search can return a single compound for exact names; for broad queries,
    # a safer approach is using "compound/name/<query>/cids" but results vary.
    url = f"{BASE_URL}/compound/name/{query}/cids/JSON"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
    cids = data.get("IdentifierList", {}).get("CID", [])
    return cids[:max_cids]

cids = get_cids_by_name(QUERY, MAX_CIDS)
len(cids), cids[:10]

### Step 3: Molecular Digitalization & Property Selection

In this section, we "pull" specific data for our list of CIDs. In cheminformatics, this process transforms a static database entry into a **feature vector**—a numerical representation of a molecule that computers can process. 

We have selected properties that are fundamental for **Drug-Likeness** (following Lipinski's Rule of Five) and future **QSAR** (Quantitative Structure-Activity Relationship) modeling:

* **Canonical SMILES:** A standardized, unique string representation of the molecular graph. It serves as the "alphabet" for computational chemical connectivity.
* **Isomeric SMILES:** Unlike Canonical SMILES, these include **stereochemical information** (chiral centers and double bond geometry), which is often the difference between an active drug and an inactive isomer.
* **InChIKey:** A fixed-length (27 character) digital signature. It is the gold standard for **deduplication**, allowing us to quickly check if the same molecule appears multiple times in our dataset.
* **Molecular Weight (MW):** A basic descriptor of molecular size. It heavily influences a drug's ability to diffuse through tissues and its renal clearance.
* **XLogP:** An algorithmic estimate of the octanol-water partition coefficient ($log P$). This measures **lipophilicity**, a key predictor of how easily a molecule crosses cell membranes.
* **TPSA (Topological Polar Surface Area):** A sum of surfaces from polar atoms (usually Nitrogen and Oxygen). It is a vital indicator of **oral bioavailability** and the ability to penetrate the blood-brain barrier (BBB).

In [ ]:
# Step 3
def fetch_properties(cids, chunk_size=50):
    """
    Fetch chemical properties from PubChem for a list of Compound IDs (CIDs).
    
    Parameters:
    -----------
    cids : list
        List of PubChem Compound IDs
    chunk_size : int
        Number of CIDs to process per batch (default: 50)
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame containing the requested properties
    """
    props = [
        "CID",
        "CanonicalSMILES",
        "IsomericSMILES",
        "InChI",
        "InChIKey",
        "MolecularWeight",
        "XLogP",
        "TPSA",
    ]
    
    all_rows = []
    total_cids = len(cids)
    
    # Validate input
    if total_cids == 0:
        print("[!] Warning: Empty CID list provided")
        return pd.DataFrame(columns=props)
    
    for i in range(0, total_cids, chunk_size):
        batch = cids[i : i + chunk_size]
        cid_str = ",".join(map(str, batch))
        
        # Construct API URL
        props_str = ",".join(props)
        url = f"{BASE_URL}/compound/cid/{cid_str}/property/{props_str}/JSON"

        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            
            batch_data = r.json().get("PropertyTable", {}).get("Properties", [])
            all_rows.extend(batch_data)
            
            # Visual feedback for progress monitoring
            retrieved = len(all_rows)
            print(f"✓ Progress: {retrieved}/{total_cids} molecules retrieved ({retrieved/total_cids*100:.1f}%)", end="\r")

        except requests.exceptions.RequestException as e:
            print(f"\n[!] Error in batch starting at index {i}: {e}")
            print(f"    Batch CIDs: {cid_str[:100]}..." if len(cid_str) > 100 else f"    Batch CIDs: {cid_str}")
            continue  # Skip this batch but continue with others
            
        except ValueError as e:
            print(f"\n[!] JSON parsing error in batch at index {i}: {e}")
            continue
            
        # Respect API rate limits
        time.sleep(0.2)
    
    # Clear the progress line and show final result
    print("\n" + " " * 80)  # Clear the line
    print(f"✓ Fetch complete: {len(all_rows)}/{total_cids} molecules successfully retrieved")
    
    # Handle case where no data was retrieved
    if not all_rows:
        print("[!] Warning: No properties were retrieved")
        return pd.DataFrame(columns=props)
    
    return pd.DataFrame(all_rows)


# Usage
df = fetch_properties(cids)
print(f"Final dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

### Step 4: Initial Data Inspection & Quality Control (QC)
"Garbage In, Garbage Out." Before moving to modeling, we must perform a sanity check on the retrieved data. This involves identifying missing values (NaNs) in critical columns and checking for structural duplicates. Even high-authority databases contain "noisy" data that must be identified early.

In [ ]:
# Step 4
df.head()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df.duplicated(subset=["CanonicalSMILES"]).sum()

### Step 5: Data Persistence and File Serialization
The final step in the acquisition phase is to export our dataframe into a persistent format. We save the data as a `.csv` file in the `data_raw` directory. 

This establishes a **data provenance** checkpoint: we preserve the original, unedited information retrieved from the API before any structural cleaning or filtering occurs in the next module. This separation between "Raw" and "Processed" data is a best practice in computational research to ensure auditability.

In [ ]:
# Step 5
out_path = os.path.join(OUT_DIR, "pubchem_raw.csv")
df.to_csv(out_path, index=False)
out_path

### Summary and Next Steps
You have successfully built a raw chemical dataset directly from the cloud. 
* **Current State:** We have a `.csv` file with 1D/2D properties.
* **Next Module:** We will use **RDKit** to perform "Chemical Curation," where we strip salts, handle tautomers, and prepare these molecules for 3D descriptor calculation.

**Suggested Exercise:** Change the `QUERY` variable to "alkaloid" or "kinase inhibitor" and re-run the notebook. Observe how the distribution of Molecular Weight and XLogP changes between chemical classes.